# Fine-Tuning and Benchmarking a Pre-trained LLM for Semantic Search (Retrieval)

This notebook is to select a pre-trained large language model (LLM), fine-tune it on a publicly available dataset
to ensure the trained model has a superior performance on e-commerce datasets, and then benchmark the
results against the selected base model.

## Selection of pre-trained models

The details of all pre-trained models that are good for semantic search is [here](https://sbert.net/docs/sentence_transformer/pretrained_models.html#semantic-search-models). I summarise the key features of a few target models below. The column 'Speed' is sentence encoding speed measured as number of sentences per second on a V100 GPU.

| Model name | Max Sequence Length | Dimensions | Suitable Score Functions | Size | Speed | Average performance |
| --- | --- | --- | --- | --- | --- | --- |
| all-mpnet-base-v2 | 384 | 768 | dot-product, cosine-similarity, euclidean distance | 420 MB | 2800 | 63.30 |
| all-MiniLM-L12-v2 | 256 | 384 | dot-product, cosine-similarity, euclidean distance | 120 MB | 7500 | 59.76 |
| all-MiniLM-L6-v2 | 256 | 384 | dot-product, cosine-similarity, euclidean distance | 80 MB | 14200 | 58.80 |

In the use case of semantic search for e-commerce, we need to find the balance between the performance and speed. From the table above, we can see that "all-mpnet-base-v2" has the best performance because it can process longer sequence and generate much higher dimension embeddings, however, it is much heavier and slower. On the other hand, "all-MiniLM-L6-v2" is much lighter and five times faster by sacrificing a little bit performance. So, I will choose "all-MiniLM-L6-v2" for fine tunning.

## Dataset selection

I will choose the [Home Depot](https://huggingface.co/datasets/Ukhushn/home-depot) dataset for fine-tuning the selected model. This dataset is a perfect candidate for fine-tunning a LLM in the context of e-commerce. It contains a number of products and real customer search terms from Home Depot's website. In the real-world application, the search algorithm needs to calculate the relevance scores between a query and products, and then responds the user with products ranked at the top for the search term. The key of the algorithm is to predict a relevance score for the provided combinations of search terms and products. To create the ground truth labels for training models, Home Depot has crowdsourced the search/product pairs to multiple human raters and then taken the average. The relevance is a number between 1 (not relevant) to 3 (highly relevant).

It should be noted that the dataset has two columns data related to products: "product_title" and "product_description". Compared with "product_title", "product_description" has more detailed description about product and thus contains much longer sentences. In this fine-tuning, I will use "product_title" first to try different settings because it is much shorter and the training using the shorter sentences will take much shorter time. If time allows, I'd like to try a few different things with "product_description: (1) use it rather than "product_title" to form ("search_term", "product_description") pairs for fine-tuning; (2) form ("search_term", "product_description") pairs and put it together with ("search_term", "product_title") pairs to double the number of data records for fine-tuning; (3) concatenate "product_title" and "product_description" to one column as "product" and then form ("search_term", "product") pairs for fine-tuning (some very long description will be truncated because the base model has a max sequence length of 256). In this notebook, using search_term and product_title for fine-tuning is perfect because they are typical short and we don't need to worry about truncation.

## Dataset preparation

How to prepare the dataset for fine-tuning depends on how we do the fine-tuning. For example, what metrics are we going to use to optimise the model during training? How are the labels distributed? For this dataset, the labels "relevance" are float numbers and the distribution is highly skewed towards the higher number end. According to the dataset card in [Home Depot](https://huggingface.co/datasets/Ukhushn/home-depot), more than 40% of labels are in the range of >= 2.6. If we use metrics for regression problems like mean squared error (MSE), we need to do oversampling for the minority to make the data balanced first, otherwise the model may have bias and predict more high scores to minimise MSE. When we fine-tune the model, we actually fine-tune the model to generate sentence embeddings and then calculate the similarity between search terms and product titles. If we use cosine similarity, the similarity scores will be in the range of [-1, 1]. As the range of the labels "relevance" are in the range of [1, 3], we need to do data scaling to match these two ranges.

However, if we use a different metrics such as MultipleNegativesRankingLoss to train the model, we don't need data balancing and label range matching. This is because MultipleNegativesRankingLoss only requires positive pairs of search terms (anchors) and product titles (positives). MultipleNegativesRankingLoss works in this way: for each batch of positive pairs, the algorithm will try to minimise the distance between the anchor and positive, and assume all other anchors' positives within the same batch as the negatives of this particular anchor and thus try to maximise the distances between this anchor and these negatives. This metrics is oftern used for semantic search with very good performance. The Home Depot dataset doesn't tell us positives and negatives directly, so we need to choose a threshold of relevance scores for positives. As more than 40% of labels are in the range of >= 2.6 and 2.6 is close to 3 indicating highly relevant, I choose 2.6 as the threhold. We need to pay attention to the assumption of this metrics: it assumes all other search terms' positives in the same batch as one search term's negatives. This means we need to take care of the duplicated search terms. The good news is we can simply avoid having the same search term in the same batch by setting an argument batch_sampler=BatchSamplers.NO_DUPLICATES. The caveat is that we cannot avoid having semantically similar search terms in the same batch, which probably is the ultimate limit of this approach.

Keep all above in mind, I use sklearn's train_test_split to split the entire dataset to training (60%), validation (20%) and test (20%) sets while maintaining the same distribution of the labels. After the split, I choose the records with relevance >= 2.6 in the training and validation sets for fine-tuning. As mentioned later, I will choose the SentenceTransformerTrainer framework for fine-tuning, and I don't need to tokenise the data because SentenceTransformerTrainer will take care of this.

In [1]:
# load the dataset
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("hf://datasets/Ukhushn/home-depot/train.csv.zip")

In [2]:
# a quick check whether the data has missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74067 entries, 0 to 74066
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   74067 non-null  int64  
 1   product_uid          74067 non-null  int64  
 2   product_title        74067 non-null  object 
 3   search_term          74067 non-null  object 
 4   relevance            74067 non-null  float64
 5   product_description  74067 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 3.4+ MB


In [3]:
# Prepare the dataset
train_text, val_test_text, train_labels, val_test_labels = train_test_split(df[['search_term', 'product_title']], 
                                                                  df['relevance'], test_size=0.4, random_state=42)
val_text, test_text, val_labels, test_labels = train_test_split(val_test_text, val_test_labels, test_size=0.5, random_state=42)

from datasets import Dataset
train_dataset = Dataset.from_pandas(df=train_text[train_labels>=2.6], preserve_index=False)
eval_dataset = Dataset.from_pandas(df=val_text[val_labels>=2.6], preserve_index=False)

## Fine-tuning

Now that the dataset is prepared for fine-tuning. We need to select a proper framework for training. I have chosen the SentenceTransformerTrainer API from sentence_transformers. With this trainer, I just need to set up the output directory "output_dir", training parameters such as the number of training epochs "num_train_epochs", training and evaluation batch sizes, learning rate, batch sampler, and tracking/debugging parameters such as evaluation strategy and steps, checkpoint save strategy, steps and number of saved checkpoints, and logging steps. Of course I need to provide the selected base model, training and evaluation datasets, as well as loss to minimise. Since the model I have chosen for fine-tuning is relatively small (66 Million parameters), I will set the learning rate = 2e-5 as the starting point.

### First attempt

In my first attempt, I just wanted to test whether the setup worked or not and how long the train would take on a CPU only laptop. Initially I thought my M2 Mac-mini with 8-core GPU might help speed up the training, but I couldn't solve the incompability issue between Pytorch and the m2 GPU chip even though I did a lot of research and tried many different things. So I moved to my relatively old Intel CPU Macbook Pro (2019 model). I set only 1 epoch for the the first trial.

In [4]:
import os
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.losses import MultipleNegativesRankingLoss

In [5]:
# first attempt of fine tuning with only 1 epoch and batch size of 16
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft1", # computer local directory to save models' checking points
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.255200,0.169325
200,0.168300,0.154893
300,0.193200,0.144707
400,0.161700,0.139407
500,0.181400,0.137025
600,0.152700,0.135498
700,0.178700,0.131724
800,0.137700,0.130546
900,0.160600,0.130622
1000,0.153600,0.126730


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=1290, training_loss=0.16626276637232582, metrics={'train_runtime': 1097.0789, 'train_samples_per_second': 18.803, 'train_steps_per_second': 1.176, 'total_flos': 0.0, 'train_loss': 0.16626276637232582, 'epoch': 1.0})

In [6]:
# save the model to local disk
model.save_pretrained(path=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft1")

### Benchmarking

It took about 18 minutes to finish the training with 1 epoch. I was really curious whether this simply fine-tuned model would be better than the base model or not. I wrote an evaluation function using NDCG@10 to benchmark the model. NDCG stands for Normalised Discounted Cumulative Gain. It is a ranking quality metrics. It compares rankings to an ideal order where all relevant items are at the top of the list. NDCG at K is determined by dividing the Discounted Cumulative Gain (DCG) by the ideal DCG representing a perfect ranking. NDCG has the range of [0, 1]. The closer to the ideal ranking the model predicted ranking is, the higher the NDCG score is and the better the model is. Since it is a ranking quality metrics, it fits this e-commerce use case very well, and we don't need to scale the similarity scores to match the relevance scores' range.

I summarise the results for the base models "all-MiniLM-L6-v2", "all-mpnet-base-v2", and fine-tuned model "all-MiniLM-L6-v2-ft1" below:

| Model name | NDCG@10 score |
| --- | --- |
| all-MiniLM-L6-v2 | 0.8766 |
| all-mpnet-base-v2 | 0.8515 |
| all-MiniLM-L6-v2-ft1 | 0.8787 |

We can see that even though the average performance of "all-mpnet-base-v2" is better than "all-MiniLM-L6-v2", "all-MiniLM-L6-v2" is slightly better on this particular dataset. The fine-tuned model "all-MiniLM-L6-v2-ft1" improved the base model by a little bit.

In [7]:
# define an evaluation function using NDCG@10
import numpy as np
from sklearn.metrics import ndcg_score
def llm_evaluation(model: SentenceTransformer, 
                   queries: np.ndarray, 
                   products: np.ndarray, 
                   labels: np.ndarray,
                   eval_metrics: str = 'NDCG@10') -> float:
    '''
    This function evaluates the performance of a llm using a particular metrics.
    
    param:
        model (SentenceTransformer): a pre-trained or fine-tuned SentenceTransformer model
        queries: a numpy array that contains the queries text
        products: a numpy array that contains the products text
        eval_metrics: a string indicating the evaluation metrics, default to NDCD@10 using ndcg_score for top 10 ranking (implemented)

    return:
        A float number
    '''
    query_embeddings = model.encode(queries)
    product_embeddings = model.encode(products)
    similarity = model.similarity_pairwise(query_embeddings, product_embeddings)
    if eval_metrics == 'NDCG@10':
        result = ndcg_score(labels, np.array(similarity).reshape(1, -1), k=10)
    return round(result, 4)

In [8]:
# benchmarking using 'sentence-transformers/all-MiniLM-L6-v2'
model1 = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
llm_evaluation(model=model1,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8766

In [9]:
# benchmarking using 'sentence-transformers/all-mpnet-base-v2'
model2 = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
llm_evaluation(model=model2,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8515

In [10]:
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8787

### Try with more epochs and larger batch size

Since the training setup has worked, next step is to increase the number of training epochs and batch sizes to see if we can improve the model further. I set num_train_epochs as 5 and double the batch size from 16 to 32. This time the training took slightly more than 1 hour. The results are summarised below:

| Model name | NDCG@10 score | settings |
| --- | --- | --- |
| all-MiniLM-L6-v2 | 0.8766 | Base model |
| all-mpnet-base-v2 | 0.8515 | Base model |
| all-MiniLM-L6-v2-ft1 | 0.8787 | epoch=1, batch size=16, lr=2e-5, no early stopping |
| all-MiniLM-L6-v2-ft2 | 0.8935 | epoch=5, batch size=32, lr=2e-5, no early stopping |

I am glad that increasing the number of training epochs and batch sizes improves the NDCG@10 score by 2% compared with the base model "all-MiniLM-L6-v2". In the training logging, we can see that the loss is not improved for a few steps and the model saved at the end of the training doesn't have the lowest loss. It is time to include an early stopping strategy.

In [11]:
# fine tuning with more epochs and larger batch size
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft2",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.388900,0.276678
200,0.309700,0.240752
300,0.284800,0.227422
400,0.285100,0.215272
500,0.256100,0.206861
600,0.245800,0.201537
700,0.221600,0.195857
800,0.206500,0.188520
900,0.186600,0.186198
1000,0.185200,0.184970


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=3225, training_loss=0.1534524398256642, metrics={'train_runtime': 3963.5201, 'train_samples_per_second': 26.022, 'train_steps_per_second': 0.814, 'total_flos': 0.0, 'train_loss': 0.1534524398256642, 'epoch': 5.0})

In [12]:
model.save_pretrained(path=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft2")

In [13]:
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8935

### Early stopping

Next I will try the early stopping strategy to see if the trainer can keep the "best model" with the lowest loss and whether it has a better NDCG@10 score. I set early_stopping_patience as 10 so that it will still run to the end (from the logging above, we can see if we set it to 5, it will stop at steps 2600). To make early stopping work, I also need to set load_best_model_at_end as True so that the best model will be loaded at the end of training. I increase save_total_limit from 2 to 6 to save more checking points.

In [14]:
# fine tuning with early stopping
from transformers.trainer_callback import EarlyStoppingCallback
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft3",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=6,
    load_best_model_at_end=True,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=10,
            early_stopping_threshold=0
        )
    ]
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.388900,0.276678
200,0.309700,0.240752
300,0.284800,0.227422
400,0.285100,0.215272
500,0.256100,0.206861
600,0.245800,0.201537
700,0.221600,0.195857
800,0.206500,0.188520
900,0.186600,0.186198
1000,0.185200,0.184970


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=3225, training_loss=0.1534524398256642, metrics={'train_runtime': 3958.7698, 'train_samples_per_second': 26.054, 'train_steps_per_second': 0.815, 'total_flos': 0.0, 'train_loss': 0.1534524398256642, 'epoch': 5.0})

Let's check whether the "best" model (the model with the lowest validation loss) has been loaded at the end of training by calculating its NDCG@10 score and comparing it with that of the model at step 2800 which has the lowest validation loss. We can see that they have the same NDCG@10 score, which means the early stopping strategy did load the model with the lowest validation loss. The bad news is the model with lowest validation loss doesn't have better NDCG@10 score.

I have also calculated the NDCG@10 scores for the models saved at a few other checking points towards the end of training and they have the same score of 0.8935. Why does the model with the lowest loss not have better performance than those with higher losses? This is probably the sign of overfitting. I will try two things to see if we can reduce overfitting by ajusting the learning rate and adding more data (using the product description data).

In [15]:
# check the loaded "best" model's performance at the end of training
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.891

In [16]:
# check the performance of the model with the lowest loss
best_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft3/checkpoint-2800")
llm_evaluation(model=best_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.891

In [17]:
# check a few other models at different checking points
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft3/checkpoint-3225")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8935

In [18]:
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft3/checkpoint-2900")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8935

In [19]:
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft3/checkpoint-3000")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8935

### Fight with overfitting - adjust learning rate

The magnitude of learning rate in transformers determines how much we adjust the weights and bias in each iteration. If it is too small, it will take a long time to converge and the training may finish before we reach the global optimal point. If it is too big, the update of weights and bias will be very aggresive and the training process may hop over the global optimal point, leading the model to converge to a local optimal point by overfitting to noise. From the training logging above, we can see that the validation loss went up and down for a few times. It is very likely that the learning rate is a little big. To prove this, I set the learning rate even bigger to see if the model's performance on test set degrades. In next experiment, I set learning rate = 1e-4 (5 times of 2e-5).

In [20]:
# fine tuning with larger learning rate
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft4",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=6,
    load_best_model_at_end=True,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0
        )
    ]
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.348700,0.240114
200,0.278500,0.225135
300,0.263700,0.220749
400,0.289100,0.214489
500,0.260000,0.208205
600,0.244600,0.204336
700,0.213500,0.188593
800,0.169000,0.180443
900,0.119200,0.180855
1000,0.097500,0.181998


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=3225, training_loss=0.09315604440001554, metrics={'train_runtime': 3839.5477, 'train_samples_per_second': 26.863, 'train_steps_per_second': 0.84, 'total_flos': 0.0, 'train_loss': 0.09315604440001554, 'epoch': 5.0})

In [21]:
# check the loaded "best" model's performance at the end of training
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8647

In [22]:
# check another model's performance at a different check point
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft4/checkpoint-3000")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8659

The results above show that increasing the learning rate with the same number of epochs dramatically reduces the training loss and validation loss, however, it cannot be generalised well on the unseen test data, clearly indicating overfitting. Next I set the learning rate = 4e-6 (1/5 of 2e-5) to see what happens.

In [23]:
# fine tuning with smaller learning rate
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft5",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=4e-6,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=6,
    load_best_model_at_end=True,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0
        )
    ]
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.415700,0.315972
200,0.359800,0.281946
300,0.329300,0.261891
400,0.325600,0.249050
500,0.298000,0.240913
600,0.292700,0.234425
700,0.266100,0.230234
800,0.259600,0.226045
900,0.263500,0.222584
1000,0.269300,0.220513


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=3225, training_loss=0.2445353272164515, metrics={'train_runtime': 3808.6572, 'train_samples_per_second': 27.08, 'train_steps_per_second': 0.847, 'total_flos': 0.0, 'train_loss': 0.2445353272164515, 'epoch': 5.0})

In [24]:
# check the loaded "best" model's performance at the end of training
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8901

In [25]:
# check the model at a different checking point
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft5/checkpoint-3000")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8901

We can see from the training logging that with a smaller learning rate, the validation loss is monolithically decreasing. The NDCG@10 score is slightly lower than 0.8935, indicating that we might be able to improve the performance by setting the learning rate somewhere between 4e-6 and 2e-5. Let's give it another go by setting learning rate = 1e-5.

In [26]:
# fine tuning with slightly bigger learning rate
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft6",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=6,
    load_best_model_at_end=True,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0
        )
    ]
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.402700,0.294630
200,0.330800,0.257305
300,0.302400,0.240535
400,0.300600,0.228621
500,0.273300,0.220291
600,0.264200,0.214089
700,0.239400,0.209729
800,0.229700,0.204285
900,0.222700,0.200707
1000,0.226600,0.199056


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=3225, training_loss=0.1953092158857242, metrics={'train_runtime': 3819.7048, 'train_samples_per_second': 27.002, 'train_steps_per_second': 0.844, 'total_flos': 0.0, 'train_loss': 0.1953092158857242, 'epoch': 5.0})

In [27]:
# check the loaded "best" model's performance at the end of training
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8945

In [28]:
# check the model at a different checking point
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft6/checkpoint-3000")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8954

In [29]:
# check the model at a different checking point
another_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft6/checkpoint-3200")
llm_evaluation(model=another_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8949

It can be seen from the results above that choosing a learning rate between 4e-6 and 2e-5 did improve the performance. I have noticed that with the same batch splitting settings, the model got the lowest validation loss at step 2800 most of the time, this is probably related to the data in that specific batch. The model saved at checking point 3000 has even better NGCD@10 score on the test set. The table below summarises all the results on test set of the base models and fine-tuned models using "product_title" data. For fine-tuned models with multiple checking points, I took the one with the lowest validation loss for a fair comparison.

| Model name | NDCG@10 score | settings |
| --- | --- | --- |
| all-MiniLM-L6-v2 | 0.8766 | Base model |
| all-mpnet-base-v2 | 0.8515 | Base model |
| all-MiniLM-L6-v2-ft1 | 0.8787 | epoch=1, batch size=16, lr=2e-5, no early stopping |
| all-MiniLM-L6-v2-ft2 | 0.8935 | epoch=5, batch size=32, lr=2e-5, no early stopping |
| all-MiniLM-L6-v2-ft3 | 0.8910 | epoch=5, batch size=32, lr=2e-5, early stopping |
| all-MiniLM-L6-v2-ft4 | 0.8647 | epoch=5, batch size=32, lr=1e-4, early stopping |
| all-MiniLM-L6-v2-ft5 | 0.8901 | epoch=5, batch size=32, lr=4e-6, early stopping |
| all-MiniLM-L6-v2-ft6 | 0.8945 | epoch=5, batch size=32, lr=1e-5, early stopping |

### Fight with overfitting - use more data

The other way to fight with overfitting is to use more data. As mentioned at the beginning, the "product_title" data is much shorter than "product_description" data. Using "product_description" data to replace "product_title" may enrich the semantical meaning of product and the LLMs may work better. However, since the sentences are much longer, both training, validation and benchmarking may take much longer time. In real-world applications, speed may be more important than a little bit performance improvement. Neverthelss, let's check what happens if we use the "product_description" data.

Before jumping straight into fine-tuning, I'd like to check the performance of base models "all-MiniLM-L6-v2" and "all-mpnet-base-v2", and a few of our best fine-tuned models "all-MiniLM-L6-v2-ft3/checkpoint-3000" and "all-MiniLM-L6-v2-ft6/checkpoint-3000" first, as summarised below:

| Model name | NDCG@10 score using product description | NDCG@10 score using product title |
| --- | --- | --- |
| all-MiniLM-L6-v2 | 0.9655 | 0.8766 |
| all-mpnet-base-v2 | 0.9914 | 0.8515 |
| all-MiniLM-L6-v2-ft3 | 0.9809 | 0.8535 |
| all-MiniLM-L6-v2-ft6 | 0.9777 | 0.8954 |

Wow, the base models' performance is really impressive, especially the bigger model "all-mpnet-base-v2". However, on my CPU only Macbook Pro 2019, it took a really long time (nearly 45 minutes) to calculate NDCG@10 score for "all-mpnet-base-v2". The interesting part is the fine-tuned "all-MiniLM-L6-v2-ft3" and "all-MiniLM-L6-v2-ft6" models are still better than its base model. "all-MiniLM-L6-v2-ft6" has better BDCG@10 score than "all-MiniLM-L6-v2-ft3" when using the product title data, but its score is slightly lower when using the product description data. This tells us a few insights: (1) These pre-trained LLMs for semantic search are really high quality if we have a lot of data; (2) fine-tuning can still improve the base model's performance; (3) when the fine-tuned model fits to the product title data so well, it will be slightly worse when generalising to different data.

In [30]:
# Use detailed product description to replace product title for fine-tuning
train_text, val_test_text, train_labels, val_test_labels = train_test_split(df[['search_term', 'product_description']], 
                                                                  df['relevance'], test_size=0.4, random_state=42)
val_text, test_text, val_labels, test_labels = train_test_split(val_test_text, val_test_labels, test_size=0.5, random_state=42)

from datasets import Dataset
train_dataset = Dataset.from_pandas(df=train_text[train_labels>=2.6], preserve_index=False)
eval_dataset = Dataset.from_pandas(df=val_text[val_labels>=2.6], preserve_index=False)

In [31]:
# benchmarking using 'sentence-transformers/all-MiniLM-L6-v2' and detailed production description data
model1 = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
llm_evaluation(model=model1,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_description']),
               labels=np.array(test_labels).reshape(1, -1))

0.9655

In [32]:
# benchmarking using 'sentence-transformers/all-mpnet-base-v2' and detailed production description data
model2 = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
llm_evaluation(model=model2,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_description']),
               labels=np.array(test_labels).reshape(1, -1))

0.9914

In [33]:
# the performance of a fine-tuned model (NDCG@10 = 0.8935 on test data using product title) on detailed production description data
best_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft3/checkpoint-3000")
llm_evaluation(model=best_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_description']),
               labels=np.array(test_labels).reshape(1, -1))

0.9809

In [34]:
# the performance of a fine-tuned model (NDCG@10 = 0.8954 on test data using product title) on detailed production description data
best_model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft6/checkpoint-3000")
llm_evaluation(model=best_model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_description']),
               labels=np.array(test_labels).reshape(1, -1))

0.9777

Now that we have the benchmark, let's do fine-tuning using the richer data by setting the learning rate=1e-5. As expected, it took much longer to train (> 6 hours).

In [35]:
# fine tuning using detailed product description and a learning rate=1e-5
from transformers.trainer_callback import EarlyStoppingCallback
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir=f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft7",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Optional tracking/debugging parameters:
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=6,
    load_best_model_at_end=True,
    logging_steps=100
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0
        )
    ]
)
trainer.train()

Step,Training Loss,Validation Loss
100,0.675900,0.541049
200,0.606200,0.485957
300,0.545800,0.451895
400,0.546900,0.426016
500,0.521500,0.410115
600,0.431500,0.397518
700,0.447100,0.389972
800,0.436400,0.381609
900,0.410200,0.375236
1000,0.396400,0.370348


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=3225, training_loss=0.35826695922733276, metrics={'train_runtime': 24437.3474, 'train_samples_per_second': 4.221, 'train_steps_per_second': 0.132, 'total_flos': 0.0, 'train_loss': 0.35826695922733276, 'epoch': 5.0})

In [36]:
# the performance of a fine-tuned model on detailed production description data
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_description']),
               labels=np.array(test_labels).reshape(1, -1))

0.9616

In [37]:
# it would be interesting to see the performance of the fine-tuned model using the product description data on the product title test set
train_text, val_test_text, train_labels, val_test_labels = train_test_split(df[['search_term', 'product_title']], 
                                                                  df['relevance'], test_size=0.4, random_state=42)
val_text, test_text, val_labels, test_labels = train_test_split(val_test_text, val_test_labels, test_size=0.5, random_state=42)
model = SentenceTransformer(f"/Users/{os.environ['USER']}/LLM/all-MiniLM-L6-v2/all-MiniLM-L6-v2-ft7/checkpoint-3225")
llm_evaluation(model=model,
               queries=np.array(test_text['search_term']),
               products=np.array(test_text['product_title']),
               labels=np.array(test_labels).reshape(1, -1))

0.8983

After nearly 7 hours training for the specific production description data, I got a model with poorer performance on the product description test set than "all-MiniLM-L6-v2-ft3" that was trained using product title data. I am a bit dissapointed but not surprised if I have a look at the training logging. It seems like the learning rate of 1e-5 is a bit too small when the dataset is bigger. The converging is really slow. If we can increase the learning rate, the model may perform better, but I won't do this experiment without a GPU.

What makes it interesting is this model has the best performance (NDCG@10 score is 0.8983) on the product title test set. This means models trained using richer data may over-perform those trained using simpler data when doing inference on unseen simpler data. If this is true in general, it will be useful when we want to have high model performance and fast inference speed at the same time.

## Conclusion and discussion

In conclusion, I have fine-tuned the base model "all-MiniLM-L6-v2" using the (search_term, product_title) and (search_term, product_description) pairs in the Home Depot data, respectively. I have selected MultipleNegativesRankingLoss as the training metrics because it makes a lot of sense in the context of semantic search for e-commerce usecases. Through adjusting the hyperparameters such as the number of train epochs, learning rate, batch size, early stopping and model saving strategies, I have improved the NDCG@10 score from 0.8766 (base model) to 0.8954 (fine-tuned model) on the test set. I have learned the following lessons during this exercise:

* When fine-tuning / training an LLM, the learning rate should be small in general as transformers have relatively complex architecture and thus a lot of parameters. But how small is small enough? For models like "all-MiniLM-L6-v2" with relatively small amount of parameters (66 Million rather than billions), when the training dataset has short sentences and tens of thousands of rows, the learning rate should probably be around 1e-5. When the dataset has very long sentences, we probably need bigger learning rate. The best way to find the optimum learning rate is to do experiments and watch how the training process converge.
* LLMs seem to work better on richer data. As we have seen here, both base models and fine-tuned models have much better NDCG@10 scores when using the more detailed product description data, even though the models are not fine-tuned using the product description data. The downside of working with richer data is the data processing speed is much slower.
* The challenge of working with LLM model is the hardware. Without a decent GPU, it is not feasible to do a lot of experiments in a short period of time. In real-world applications where the response latency is very important, we would also need GPUs to speed up the inference process.
* It seems that models trained using richer data may over-perform those trained using simpler data when doing inference on unseen simpler data. If this is true in general, it will be useful when we want to have high model performance and fast inference speed at the same time.

At last, I'd like to summarise the results of models with the best performance on test set in the table below:

| Model name | NDCG@10 score using product description | NDCG@10 score using product title | settings | training data |
| --- | --- | --- | --- | --- |
| all-MiniLM-L6-v2 | 0.9655 | 0.8766 | Base model | Public |
| all-mpnet-base-v2 | 0.9914 | 0.8515 | Base model | Public |
| all-MiniLM-L6-v2-ft3 | 0.9809 | 0.8535 | epoch=5, batch size=32, lr=2e-5, early stopping | Home Depot: product title |
| all-MiniLM-L6-v2-ft6 | 0.9777 | 0.8954 | epoch=5, batch size=32, lr=1e-5, early stopping | Home Depot: product title |
| all-MiniLM-L6-v2-ft7 | 0.9616 | 0.8983 | epoch=5, batch size=32, lr=1e-5, early stopping | Home Depot: product description |